<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 5 (3): CrewAI — Agents With Job Titles

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Describe an agent by its **role, goal and backstory** instead of a prompt
2. Run your first **Agent + Task + Crew**
3. Split work between **two specialists** and watch one hand off to the other
4. Give a crew **tools**
5. Turn on **memory** so later tasks remember earlier ones
6. Decide **which of the three approaches** you have now seen actually fits a given job

> **Notebooks 1 and 2 first.** You already know what a tool call is and what an agent loop
> does. CrewAI is the same machinery again — with the loop hidden and the *roles* in front.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q crewai

In [ ]:
import os
from getpass import getpass
from IPython.display import Markdown, display

from crewai import Agent, Task, Crew, Process, LLM

api_key = getpass("Enter your OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = api_key

print("Setup complete")

In [ ]:
# CrewAI's LLM class takes a "provider/model" string.
# To use a different provider you change this one line - nothing else.
#
# NOTE: run the cell above FIRST. LLM() checks for the key as soon as you create it,
# so running this out of order gives you "OPENAI_API_KEY is required".
llm = LLM(model="openai/gpt-4o-mini")

# llm = LLM(model="gemini/gemini-2.0-flash")          # needs GEMINI_API_KEY
# llm = LLM(model="groq/llama-3.3-70b-versatile")     # needs GROQ_API_KEY

print(f"LLM configured: {llm.model}")

---

## 2. Why Describe Agents by Role?

You have now built the same thing twice:

| | You wrote | You managed |
|---|---|---|
| **Notebook 1** | the loop, the schemas, the message list | everything |
| **Notebook 2** | nodes, edges, state | the graph |
| **Notebook 3** | *a job description* | nothing |

CrewAI takes a different angle. Instead of describing the **machinery**, you describe the **people**:

```
  AGENT  = role + goal + backstory       "who is doing this, and why"
  TASK   = description + expected_output "what needs doing, and what done looks like"
  CREW   = agents + tasks                "put them together and go"
```

The loop is still there. `finish_reason` is still doing its job somewhere underneath. You just stop
looking at it.

> This is the trade every framework makes: **less control, less code**. Notebook 1 gave you total
> control and total responsibility. This gives you a paragraph of English and a working crew.

---

## 3. Your First Agent, Task and Crew

Three objects, in order. Start with the **agent** - who is doing the work.

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="Find accurate, useful information on any topic",
    backstory="You are a skilled researcher who finds key facts quickly and summarises them clearly.",
    llm=llm,
    verbose=True,          # print the agent's thinking - keep this on while learning
)

print(f"Agent created - role: {researcher.role}")

Now the **task** - what needs doing. Note `expected_output`: you are describing what *done*
looks like, which is what stops the agent wandering.

In [ ]:
research_task = Task(
    description="Research the main applications of AI in healthcare. Find 3 key applications.",
    expected_output="A list of 3 AI applications in healthcare, each with a one-sentence explanation.",
    agent=researcher,      # who does this task
)

print("Task created")

And the **crew** - put them together and run it. `kickoff()` is the equivalent of `invoke()`.

In [ ]:
crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    verbose=True,
)

result = crew.kickoff()

print("\nFinal output:")
display(Markdown(str(result)))

Scroll back through that `verbose` output. You will see the agent state what it is doing and
then produce a final answer - the same think-then-answer cycle from notebook 1, just narrated.

---

## 4. Anatomy — Role, Goal, Backstory

Those three fields are not decoration. They are **the system prompt, split into three questions**:

| Field | The question it answers | Keep it |
|---|---|---|
| `role` | *Who are you?* | short - a job title |
| `goal` | *What are you trying to achieve?* | one sentence, outcome-focused |
| `backstory` | *Why are you good at this, and how do you work?* | two or three sentences of style and priorities |

And on the task:

| Field | The question it answers |
|---|---|
| `description` | *What exactly needs doing?* |
| `expected_output` | *How will we know it is finished?* |

💡 **`expected_output` is the field people skip and then regret.** A vague one gives you a rambling
answer; a specific one ("a list of 3, each one sentence") gives you something you can use. It is the
same lesson as the tool `description` in notebook 1 - the English **is** the engineering.

---

## 5. Two Specialists — a Real Crew

One agent doing everything is just an agent. A **crew** is several agents with different jobs,
passing work along.

Here: a researcher who finds facts, and a writer who turns them into prose. Neither is asked to do
the other's job.

In [ ]:
researcher = Agent(
    role="Research Specialist",
    goal="Find accurate, relevant and comprehensive information on any topic",
    backstory=(
        "You are an experienced researcher with a talent for finding key information quickly. "
        "You focus on facts and evidence, and you present findings in a clear, organised way."
    ),
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Content Writer",
    goal="Turn research into clear, engaging, well-structured writing",
    backstory=(
        "You are a skilled writer who makes complex topics easy to understand. "
        "You write in a clear, conversational style and always structure your work well."
    ),
    llm=llm,
    verbose=True,
)

print(f"Crew members: {researcher.role}, {writer.role}")

Two tasks, each assigned to the specialist who should do it. The writer's task refers to
"the research provided" - CrewAI passes the first task's output into the second.

In [ ]:
research_task = Task(
    description=(
        "Research 'sustainable living practices'. Find 4 practices an individual can adopt, "
        "and note why each one matters."
    ),
    expected_output="A list of 4 sustainable living practices with a short explanation of each.",
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research provided, write a short blog post about sustainable living. "
        "Make it engaging and practical. Keep it under 250 words."
    ),
    expected_output="An engaging blog post under 250 words.",
    agent=writer,
)

content_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,     # run tasks in order, passing output along
    verbose=True,
)

result = content_crew.kickoff()

print("\nFinal blog post:")
display(Markdown(str(result)))

**Watch the handoff in the verbose output.** The researcher finishes, its output becomes the
writer's input, and the writer never had to do any research.

`Process.sequential` is what you will use almost always: tasks run in order, each seeing what came
before. (There is also `Process.hierarchical`, where a manager agent decides who does what - more
impressive, harder to predict, and rarely what you need.)

⚠️ **The honest warning.** Two agents cost roughly twice as much as one, and - the part people miss -
**context does not cross the boundary for free**. The writer only knows what the researcher wrote
down, and passing that along costs tokens on every step.

> **Default to one agent with more tools.** Split into a crew only when the jobs genuinely need
> different skills, different tools, or different permissions.

---

## 6. Giving a Crew Tools

Same idea as notebook 1: the agent can only act if you hand it functions. CrewAI has its own
`@tool` decorator, and - as always - **the docstring is what the model reads**.

In [ ]:
from crewai.tools import tool


@tool("Company document search")
def search_company_docs(query: str) -> str:
    """Search internal company documents for policies, benefits and product pricing.

    Args:
        query: what to search for

    Returns:
        The matching policy text.
    """
    documents = {
        "leave": "Employees receive 24 paid leaves and 10 sick leaves per year.",
        "notice": "Notice period is 2 months for permanent employees, 1 month during probation.",
        "learning": "The learning budget is 50,000 per year per employee.",
        "insurance": "Health insurance covers 5 lakh for employees and their families.",
    }
    for key, value in documents.items():
        if key in query.lower():
            return value
    return "No matching policy found. Say that you do not know."


print(f"Tool ready: {search_company_docs.name}")

In [ ]:
hr_agent = Agent(
    role="HR Assistant",
    goal="Answer employee questions using only the official company documents",
    backstory=(
        "You are a careful HR assistant. You always look up the policy before answering, "
        "and you never guess a number."
    ),
    tools=[search_company_docs],       # <- the only new argument
    llm=llm,
    verbose=True,
)

hr_task = Task(
    description="An employee asks: how many paid leaves do I get, and what is the learning budget?",
    expected_output="A short answer stating both numbers, taken from the documents.",
    agent=hr_agent,
)

hr_crew = Crew(agents=[hr_agent], tasks=[hr_task], verbose=True)

result = hr_crew.kickoff()

print("\nAnswer:")
display(Markdown(str(result)))

In the verbose output you can see the agent decide to use the tool, call it, read the result,
and only then answer. That is the loop from notebook 1 - you just never had to write it.

---

## 7. Memory

By default each crew run starts fresh. `memory=True` lets later tasks draw on what happened
earlier - the same idea as the `thread_id` in notebook 2, with the plumbing hidden.

⚠️ Memory is not free: CrewAI stores past steps as **embeddings**, so switching it on adds embedding
calls to your bill and a little latency. Turn it on when tasks genuinely depend on each other, not by
reflex.

In [ ]:
assistant = Agent(
    role="HR Assistant",
    goal="Answer employee questions and remember what has already been discussed",
    backstory="You are a helpful assistant who keeps track of the conversation so far.",
    tools=[search_company_docs],
    llm=llm,
    verbose=True,
)

first_task = Task(
    description="Look up the notice period for permanent employees.",
    expected_output="The notice period, stated plainly.",
    agent=assistant,
)

second_task = Task(
    description="An employee resigning today asks how much notice they must serve. Use what you already found.",
    expected_output="A one-sentence answer that reuses the earlier lookup instead of searching again.",
    agent=assistant,
)

memory_crew = Crew(
    agents=[assistant],
    tasks=[first_task, second_task],
    process=Process.sequential,
    memory=True,                     # <- the only new argument
    verbose=True,
)

result = memory_crew.kickoff()

print("\nFinal answer:")
display(Markdown(str(result)))

---

## 8. Which One Should You Actually Use?

You have now built the same capability three times. That was deliberate - here is the comparison you
could not have understood on Monday.

| | **Raw loop** *(nb 1)* | **LangGraph** *(nb 2)* | **CrewAI** *(nb 3)* |
|---|---|---|---|
| You write | the loop, schemas, messages | state, nodes, edges | role, goal, backstory |
| Lines for a simple agent | ~40 | ~15 | ~10 |
| Control | total | high | low |
| Can you draw it? | no | **yes** | not really |
| Pause / resume | you build it | **built in** | no |
| Best at | learning, and anything unusual | production agents, branching, approvals | quick multi-role drafts |
| Debugging | print statements | the graph + state history | reading verbose logs |

**How I would choose:**

- **One tool, one job?** Don't use a framework. Notebook 1's loop is 40 lines and you own all of it.
- **Branching, approvals, memory, anything going to real users?** **LangGraph.** The graph is a
  design document that happens to run, and `interrupt_before` is not something you want to rebuild.
- **Several roles drafting something together, and you want it working this afternoon?** **CrewAI.**
- **Still deciding?** Re-read notebook 2 section 5. Most of the time the honest answer is that you
  do not need an agent at all - you need a workflow with fixed steps.

> Frameworks are not levels of skill. They are different trades between control and code. Knowing
> what each one hides is the actual skill - and now you have seen underneath all three.

---

## 9. Exercises

Fill in the blanks (`___`). Hints are in the comments.

### Q1: Your own single agent

Build one agent that writes a short product description for a topic you choose.

In [ ]:
my_agent = Agent(
    role="___",                    # a job title, two or three words
    goal="___",                    # one sentence, outcome-focused
    backstory="___",               # why they are good at this, and how they work
    llm=llm,
    verbose=True,
)

my_task = Task(
    description="___",
    expected_output="___",         # what does DONE look like? be specific
    agent=___,
)

print(Crew(agents=[my_agent], tasks=[my_task], verbose=True).kickoff())

### Q2: Add an editor

Extend the researcher + writer crew with a third agent that shortens the post and fixes the tone.

In [ ]:
editor = Agent(
    role="___",
    goal="___",
    backstory="___",
    llm=llm,
    verbose=True,
)

edit_task = Task(
    description="Edit the blog post: cut it to 150 words and make the tone friendlier.",
    expected_output="___",
    agent=___,
)

three_crew = Crew(
    agents=[researcher, writer, ___],
    tasks=[research_task, writing_task, ___],
    process=Process.___,
    verbose=True,
)

display(Markdown(str(three_crew.kickoff())))

### Q3: A tool of your own

Write a tool the agent cannot answer without, and prove it gets used.

In [ ]:
@tool("___")
def get_stock_level(product: str) -> str:
    """___                                  # this docstring is what the model reads

    Args:
        product: ___

    Returns:
        ___
    """
    return "___"


# Hint: give it to an Agent via tools=[...], then write a task that cannot be
# answered without calling it. Check the verbose output to confirm it was used.

### Q4: Argue the choice

No code. In a comment, pick one of these and justify it in two sentences using the table in
section 8:

- a bot that answers HR policy questions for 500 employees
- a one-off script that turns your notes into a summary
- an internal tool that can issue refunds up to 5,000

In [ ]:
# Which approach for which, and why?
#
# HR policy bot        -> ___ because ___
# One-off summariser   -> ___ because ___
# Refund tool          -> ___ because ___

---

## Key Takeaways

1. **CrewAI describes people, not machinery.** Role, goal and backstory are a system prompt split
   into three questions.

2. **Agent + Task + Crew.** The agent is who, the task is what, the crew puts them together and
   `kickoff()` runs it.

3. **`expected_output` is doing real work.** Vague in, rambling out. It is the same lesson as the
   tool description in notebook 1: the English is the engineering.

4. **`Process.sequential` passes output along.** Each task sees what the previous one produced -
   that is the handoff.

5. **Tools work the same way everywhere.** A function plus a docstring the model reads. Third
   notebook, third syntax, one idea.

6. **Multi-agent costs more than it looks.** Twice the agents is roughly twice the bill, and context
   does not cross the boundary for free. Default to one agent with more tools.

7. **Three frameworks, one trade.** Control versus code. Knowing what each one hides is the skill.

### Concept Map

```
   +---------------------------- CREW ----------------------------+
   |                                                              |
   |   AGENT (researcher)            AGENT (writer)               |
   |   role / goal / backstory       role / goal / backstory      |
   |   tools=[...]                                                |
   |        |                             ^                       |
   |        v                             |                       |
   |   TASK 1  --------- output ----------+                       |
   |   description + expected_output                              |
   |                                                              |
   |   process=Process.sequential      memory=True                |
   +--------------------------------------------------------------+
                              |
                          kickoff()
                              |
                         final result
```

### Quick Reference

| Piece | What it is |
|---|---|
| `LLM(model="openai/gpt-4o-mini")` | which model; change the string to change provider |
| `Agent(role, goal, backstory, llm)` | who is doing the work |
| `tools=[...]` on an Agent | what it is allowed to do |
| `Task(description, expected_output, agent)` | what needs doing, and what done looks like |
| `Crew(agents, tasks)` | put them together |
| `Process.sequential` | run tasks in order, passing output along |
| `memory=True` | later tasks can draw on earlier ones |
| `verbose=True` | print the reasoning — keep it on while learning |
| `crew.kickoff()` | run it |
| `@tool("name")` | a function the agents can call; the docstring is the description |

### 🏠 Homework

1. **A crew for something you actually do.** Two or three agents, a real task from your own life or
   coursework. Run it and keep the output.
2. **Cut it down.** Rewrite that same crew as a **single** agent with the right tools. Which output
   was better, and which cost less? Write down both answers.
3. **Justify a choice.** For your capstone idea, pick raw loop, LangGraph or CrewAI and write a
   short paragraph defending it against the other two.

### 📚 Resources

- [CrewAI documentation](https://docs.crewai.com/)
- [CrewAI — tools](https://docs.crewai.com/concepts/tools)
- [Anthropic — Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

---

**That's Week 1.** Day 1 the model answered · Day 2 you learned to steer it with prompting ·
Day 3 you made text searchable by meaning · Day 4 it answered only from your documents, with
citations · Day 5 it **acts** — and you know what that costs, how to check it, and where to put a
human in the way.

**Next week:** the AI goes behind an API you build yourself.